# Spectral deconvolution and post-processing

*Marco Berzborn (1), Simon Kersten (2)*<br>  
*(1) Department of the Built Environment, Eindhoven University of Technology*<br>  
*(2) Institute for Hearing Technology and Acoustics, RWTH Aachen University*<br> 

*Contact: m.berzborn@tue.nl, simon.kersten@akustik.rwth-aachen.de*

In this notebook, you will compute a room impulse response from a measured sweep response and improve its quality in post-processing. Starting from a recorded exponential sweep, you will compare direct and regularized spectral deconvolution, use averaging of repeated measurements to increase the signal-to-noise ratio, locate the harmonic distortion products caused by the loudspeaker's nonlinearities, and apply a time window to remove noise and distortion products. Throughout the notebook, you will inspect the effects in the time and frequency domain.


**Duration:** 60 Minutes

**Requirements:** Familiarity with digital signal processing fundamentals.

**References**<br>  
Farina, A. (2000). *Simultaneous measurement of impulse response and distortion with a swept-sine technique* (Preprint No. 5093). 108th Audio Engineering Society Convention, Paris, France. https://www.aes.org/e-lib/browse.cfm?elib=10211<br>  
Kirkeby, O., & Nelson, P. A. (1999). Digital filter design for inversion problems in sound reproduction. *Journal of the Audio Engineering Society, 47*(7/8), 583–595. https://www.aes.org/e-lib/browse.cfm?elib=12098<br>  
Müller, S., & Massarani, P. (2001). *Transfer-function measurement with sweeps: Director's cut including previously unreleased material and some corrections*. Original work published in *Journal of the Audio Engineering Society, 49*(6), 443–471.<br>  
Norcross, S. G., Bouchard, M., & Soulodre, G. A. (2006). *Inverse filtering design using a minimal-phase target function from regularization* (Convention Paper No. 6929). 121st Audio Engineering Society Convention, San Francisco, CA, United States.  

**Dependencies**<br>  
`pip install pyfar>=0.8 pooch ipympl watermark`

In [ ]:
import pyfar as pf
import numpy as np
import matplotlib.pyplot as plt
import pooch
import os

%matplotlib ipympl

## Preparation
Read the necessary files for the excitation signal and the recorded sweep response. If they are not already in the same folder as this notebook, they are automatically downloaded to this folder using `pooch`. The following code cell loads them into `pyfar.Signal` objects. The excitation signal is an exponential sweep, which is used to excite the system under test. The recorded sweep response contains the system's response to the excitation signal.

In [ ]:
# Load data
# Do not modify the following code cell. It downloads the necessary files to
# the folder of this notebook, unless they are already there.
url = (
    "https://github.com/pyfar/files/raw/refs/heads/main/education/"
    "ASSA2026_Experimental/"
)

file_excitation = pooch.retrieve(
    url + "excitation_sweep_fft18.wav",
    "ddade3746f6068417e43a40b2c14e183526534c04ace4ff932c5e402443c3c98",
    fname="excitation_sweep_fft18.wav",
    path=os.getcwd(),
)
file_recording = pooch.retrieve(
    url + "rec_sweep_3reps_fft18.wav",
    "2000c09a39d32a241ba4f5202231a6e9b7e047c0e8740185e550d7b8911d5d95",
    fname="rec_sweep_3reps_fft18.wav",
    path=os.getcwd(),
)

s_sweep = pf.io.read_audio(file_excitation)
y_sweeps = pf.io.read_audio(file_recording)

The following cell defines some variables which contain information about the used sweep excitation signal. Do not change the contents of this cell. You can use them in the following tasks.

In [ ]:
y_sweep = y_sweeps[0]
frequency_range_sweep = [20, 20e3]

n_samples_sweep = 2**18
length_sweep = n_samples_sweep / s_sweep.sampling_rate

sampling_rate = s_sweep.sampling_rate

## Spectral deconvolution

The naive approach to compute the impulse response would be to do an inverse convolution in the time domain. However, it is more efficient and flexible to do a frequency domain deconvolution (Müller & Massarani, 2001). Applying the Fourier Transform to the convolution $y[k] = s[k] \ast h[k]$ yields

$$Y[\mu] = H[\mu] S[\mu]$$

with capital letters denoting complex spectra and $\mu$ the index of the discrete frequency bins. The transfer function (Fourier Transform of the impulse response) can now be computed as

$$H[\mu] = \frac{Y[\mu]}{S[\mu]} = Y[\mu]\cdot G[\mu]\,.$$

While this looks simple, it requires the direct inversion of the excitation signal, $G[\mu] = 1/S[\mu]$, which should often be avoided.


### Regularized spectrum inversion

Since the excitation signal is bandlimited, the direct inversion will boost out of band noise, which you can see in the plot generated in the task below. To avoid this, a regularized inversion can be applied (Kirkeby & Nelson, 1999; Norcross et al., 2006). The regularized inversion is defined as

$$G_\lambda[\mu] = \frac{S^*[\mu]}{|S[\mu]|^2 + \lambda[\mu]}, \qquad \lambda[\mu] = \beta\,|\epsilon[\mu]|^2$$

with the complex conjugate $^*$, the regularization parameter $\lambda[\mu]$, a scaling factor $\beta$, and the frequency-dependent regularization function $\epsilon[\mu]$.

**Task:** Implement the regularized inversion of the excitation signal. Use the [`pyfar.dsp.RegularizedSpectrumInversion` class](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.RegularizedSpectrumInversion) to compute the regularized inversion of the excitation signal. Use the `from_frequency_range` method to specify the frequency range of interest. Use the `invert` method to compute the regularized inversion. The frequency range of the regularization is independent of the frequency range of the sweep, which is given by `frequency_range_sweep`. Execute the cell below for different frequency ranges of the regularization, for example ranges that are narrower or wider than that of the sweep, and investigate how the choice affects the inversion. Which range is appropriate for this excitation signal? Store the inverted signal in the variable `s_sweep_inverted`, which is used by the plot below and in the following tasks.

**Hint:** Use the option `beta='energy'`. This computes the scaling factor $\beta$ from the energies of the excitation signal and the regularization function, $|S[\mu]|^2$ and  $|\epsilon[\mu]|^2$, such that both summands in the denominator of the equation above are of similar magnitude.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.figure()
ax = pf.plot.freq(1 / s_sweep, label="Sweep, direct inversion", color="grey")
pf.plot.freq(
    s_sweep_inverted, label="Sweep, regularized inversion", color="C0",
)
ax.legend(loc="upper left")
plt.show()

### Calculation of the impulse response
The transfer function is now computed by a spectral multiplication of the recorded sweep spectrum with the carefully inverted excitation signal $G_\lambda[\mu]$

$$\hat H[\mu]=Y[\mu]\,G_\lambda[\mu]$$

and the impulse response is given by the inverse Fourier Transform

$$\hat h[k]=\mathrm{F}^{-1}\{\hat H[\mu]\}\,.$$

**Task:** Calculate the impulse response for both the direct and regularized inversion. Compare the results in the plot provided below and discuss the differences in the time and frequency domains. Use the single repetition `y_sweep` and store the results in the variables `h_sweep_direct` and `h_sweep_regularized`, which are used by the plot below.

**Hint:** In pyfar, the frequency domain multiplication is overloaded by the `*` operator and can thus elegantly be realized in a single line.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.figure()
ax = pf.plot.time_freq(
    pf.utils.concatenate_channels(
        [
            h_sweep_direct,
            h_sweep_regularized,
        ],
    ),
    dB_time=True,
    label=[
        "Direct inversion",
        "Regularized inversion",
    ],
)
ax[1].legend()
plt.show()

## Averaging of repeated measurements
The recorded signal has three channels, which correspond to three consecutive repetitions of the same measurement. The impulse response can be improved by averaging over the three channels.

Averaging $M$ synchronized repetitions leaves the time-invariant system response unchanged, while the noise power reduces to $1/M$, as long as the noise is independent between repetitions. The expected SNR gain is therefore (Müller & Massarani, 2001)

$$\Delta\text{SNR} = 10\log_{10} M \;\text{dB}\,.$$

**Task:** Implement averaging over the channels, calculate the expected improvement in SNR and compare your expectation with the achievable improvement in SNR. Consider the following: Does it make a difference if the averaging operation is performed on the recorded signal or on the computed impulse response? Store the expected improvement in dB in the variable `Delta_SNR`, the impulse response of the single repetition `y_sweep` in `h_sweep_single`, and the averaged impulse response in `h_sweep_avg`. These variables are used by the code below, and `h_sweep_avg` is also used in the following tasks.

**Hint:** Plot the impulse responses of the individual channels. The noise floor is not the same in all repetitions and it also varies over time, which affects the SNR improvement you obtain when comparing against a single repetition.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.figure()
print(f"Expected SNR improvement: {Delta_SNR:.2f} dB")
pf.plot.time(h_sweep_single, label="Single sweep response", dB=True)
pf.plot.time(h_sweep_avg, label="Averaged sweep response", dB=True)
plt.legend()
plt.show()

**Task:** Create a plot, which visualizes the required number of repetitions needed for a certain SNR improvement. How many repetitions are needed to achieve an SNR improvement of 15 dB? How long would the measurement take, considering the length of the excitation signal used in this experiment? 

**Hint**: One repetition lasts as long as the excitation signal `s_sweep` including its padded zeros, that is `s_sweep.signal_length`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

## Nonlinearities

The loudspeaker used during the measurement is weakly nonlinear and distortion products are expected to appear in the measured impulse response as additional impulse responses. For an exponential sine sweep, their temporal positions can be calculated as (Farina, 2000)

$$\Delta t_\ell = \Lambda \ln \ell, \qquad \Lambda = \frac{T}{\ln(f_2/f_1)}$$

with the harmonic order $\ell$, the time constant $\Lambda$ of the sweep, which is given by the sweep duration $T$, and the start and stop frequencies $f_1$ and $f_2$. $\Delta t_\ell$ is the time by which the harmonic impulse response of order $\ell$ precedes the linear impulse response.

**Note:** Due to the periodicity of the discrete Fourier transform, these harmonic impulse responses appear to be shifted to the end of the measured impulse response instead of preceding the linear impulse response.

**Task:** Calculate the temporal positions where harmonic distortion products are expected to appear in the measured impulse response and extend the code below. You can check your results by executing the code cell and plotting the measured impulse response with the expected positions of the harmonic distortion products. Store the harmonic orders in the variable `orders`, their expected positions in seconds in `t_harmonics`, and the averaged impulse response to be plotted in `h_sweep_shifted`.

**Hints:** For the calculation of the time constant of the sweep, the length of the actual sweep is to be used. The excitation signal `s_sweep` contains padded zeros, which are not part of the actual sweep. The length of the actual sweep is already given by the variable `length_sweep`. The measured impulse response further contains a delay, caused by the propagation time from the loudspeaker to the microphone and the latency of the audio interface. The delay can be estimated using the [`pyfar.dsp.find_impulse_response_delay` function](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.find_impulse_response_delay). The measured impulse response can then be shifted in time using the [`pyfar.dsp.time_shift` function](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.time_shift). Alternatively, the expected time positions of the harmonic distortion products can be shifted by the estimated delay. In this case, the impulse response remains unshifted, that is `h_sweep_shifted = h_sweep_avg`.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.figure()
ax = pf.plot.time(h_sweep_shifted, label="Averaged sweep response", dB=True)

for order, t in zip(orders, t_harmonics, strict=True):
    suffix = {2: "nd", 3: "rd"}.get(order, "th")
    ax.axvline(
        x=t, color=f"C{order}", ls="-.", label=f"{order}{suffix} harmonic"
    )
ax.legend()
plt.show()


## Time windowing

Before the onset of the impulse response and after it has decayed into the noise floor, the samples mainly contain noise and, towards the end, the harmonic impulse responses. A time window removes these parts by fading in shortly before the onset and fading out after the decay. The fade avoids the discontinuity that abruptly cutting the impulse response would cause, which would smear energy over frequency. The window also presents a trade-off: if it is too long, noise remains in the impulse response, and if it is too short, the decay is cut off, which affects the spectrum, in particular its fine structure and low frequencies.

**Task:** Inspect the averaged impulse response and choose where it starts and where it reaches the noise floor. Use [`pyfar.dsp.time_window`](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.time_window) to apply a window that fades in before the onset and fades out after the decay into the noise floor. Compare the windowed and unwindowed impulse responses in the time and frequency domain. Is the window capable of removing the noise entirely while preserving the important characteristics of the impulse response? Store the result in the variable `h_sweep_avg_windowed`, which is used by the plot below.

**Hint:** Use the [pyfar interactive shortcuts](https://pyfar.readthedocs.io/en/stable/modules/pyfar.plot.html#pyfar.plot.shortcuts) to zoom in on the impulse response.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

plt.figure()
ax = pf.plot.time_freq(
    pf.utils.concatenate_channels(
        [
            h_sweep_avg,
            h_sweep_avg_windowed,
        ],
    ),
    dB_time=True,
    label=[
        "Averaged sweep response",
        "Averaged sweep response, windowed",
    ],
)
ax[1].legend()
plt.show()


# License notice

This notebook © **2026** by **Marco Berzborn and Simon Kersten** is licensed under [CC BY 4.0](http://creativecommons.org/licenses/by/4.0/?ref=chooser-v1)

# Watermark

The following watermark might help others to install specific package versions that might be required to run the notebook. Please give at least the versions of Python, IPython, numpy , and scipy, major third party packagers (e.g., pytorch), and all used pyfar packages.

In [ ]:
%load_ext watermark
%watermark -v -m -p numpy,scipy,pyfar,sofar,watermark